# 🤖 ROTBOTS — Script Generator
## From Sources to Storyboard

---

This notebook takes you through the first half of the ROTBOTS pipeline:

1. **Feed the Machine** — Paste URLs or text, the system scrapes and summarizes them
2. **The Script Writer** — An LLM generates a structured essay with narration and visual directions
3. **Visual Translation** — The essay becomes a storyboard with styles and text-to-video prompts

**You don't need to understand the code.** Just fill in your inputs and press ▶️ Play on each cell.

---

## 🔧 Station 0: Setup

In [ ]:
# ============================================================
# SETUP
# ============================================================

!pip install -q requests beautifulsoup4 pymupdf

import json
import re
import random
import requests
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, JSON, HTML

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Setup complete')
print(f'📁 Workspace: {WORK_DIR}')

In [ ]:
# ============================================================
# API KEY — Paste your Groq API key here (starts with gsk_)
# Get one free at: https://console.groq.com/keys
# ============================================================

GROQ_API_KEY = ''  # <-- Paste your key here

LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'  # DO NOT put key here

if not GROQ_API_KEY: print('⚠️  Paste your Groq API key above!')
elif not GROQ_API_KEY.startswith('gsk_'): print('⚠️  Key should start with gsk_')
else: print(f'✅ API key configured ({LLM_MODEL})')

In [ ]:
# ============================================================
# VIDEO LENGTH SETTINGS — This controls the ENTIRE pipeline!
# ============================================================

TARGET_VIDEO_LENGTH = 30  # seconds (total video length target)
SECONDS_PER_SCENE = 5     # each AI-generated clip is ~5 seconds

# Auto-calculate structure from target length
num_content_scenes = max(2, TARGET_VIDEO_LENGTH // SECONDS_PER_SCENE)
NUM_CHAPTERS = max(1, min(3, num_content_scenes // 2))
SECTIONS_PER_CHAPTER = max(1, num_content_scenes // NUM_CHAPTERS)

# Words per scene: ~5 seconds of speech = ~12-15 words (slow narration pace)
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5  # ~2.5 words/second for clear narration

print(f'🎬 Target video length: {TARGET_VIDEO_LENGTH}s')
print(f'   {num_content_scenes} content scenes × {SECONDS_PER_SCENE}s each')
print(f'   {NUM_CHAPTERS} chapters × {SECTIONS_PER_CHAPTER} sections')
print(f'   ~{int(WORDS_PER_SCENE)} words per scene narration (~{SECONDS_PER_SCENE}s speech)')
print(f'   + title card + credits')

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    payload = {'model': LLM_MODEL, 'messages': messages, 'temperature': temperature or LLM_TEMPERATURE, 'max_tokens': LLM_MAX_TOKENS}
    response = requests.post(GROQ_API_URL, headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'}, json=payload, timeout=120)
    response.raise_for_status()
    content = response.json()['choices'][0]['message']['content']
    if '<think>' in content and '</think>' in content:
        content = content.split('</think>')[-1].strip()
    return content

def parse_json_response(response):
    response = response.strip()
    response = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response)
    if response.startswith('```'):
        lines = response.split('\n')
        response = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    response = response.strip()
    if not response.startswith('[') and not response.startswith('{'):
        for ch in ['{', '[']:
            idx = response.find(ch)
            if idx != -1: response = response[idx:]; break
    for end_char in ['}', ']']:
        last_idx = response.rfind(end_char)
        if last_idx != -1:
            truncated = response[:last_idx + 1]
            for text in [truncated, re.sub(r',\s*([}\]])', r'\1', truncated)]:
                try: return json.loads(text, strict=False)
                except json.JSONDecodeError: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\1', response), strict=False)

def fetch_website_text(url, max_chars=10000):
    url = url.strip().rstrip('/').strip()
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article','main','[role="main"]','.content','#content','.post']:
        main = soup.select_one(sel)
        if main: break
    text = main.get_text(separator=' ',strip=True) if main else (soup.find('body') or soup).get_text(separator=' ',strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

def fetch_pdf_text(source, max_chars=10000):
    import tempfile
    try: import pymupdf as fitz
    except ImportError: import fitz
    temp_file = None
    try:
        if source.startswith('http'):
            resp = requests.get(source, headers={'User-Agent':'Mozilla/5.0'}, timeout=60); resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False); temp_file.write(resp.content); temp_file.close()
            pdf_path = temp_file.name
        else: pdf_path = source
        doc = fitz.open(pdf_path); parts = []; total = 0
        for page in doc:
            t = page.get_text(); parts.append(t); total += len(t)
            if total >= max_chars: break
        doc.close()
        return re.sub(r'\n{3,}','\n\n','\n'.join(parts))[:max_chars]
    finally:
        if temp_file: import os; os.unlink(temp_file.name)

print('✅ Helper functions loaded')

---
## 📥 Station 1: Feed the Machine

> **🤔 Think about:** What gets lost when the machine compresses your sources?

In [ ]:
# YOUR INPUTS
TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
]

In [ ]:
# SCRAPE SOURCES
print(f'📥 Processing {len(SOURCES)} source(s)...')
source_texts = []
for i, source in enumerate(SOURCES):
    print(f'\n🔗 Source {i+1}: {source[:80]}')
    try:
        if source.startswith('http') and source.lower().endswith('.pdf'): text = fetch_pdf_text(source)
        elif source.startswith('http'): text = fetch_website_text(source)
        elif source.endswith('.pdf'): text = fetch_pdf_text(source)
        else: text = source
        print(f'   Extracted: {len(text)} chars')
        source_texts.append({'source': source[:100], 'text': text})
    except Exception as e: print(f'   ❌ {e}')
print(f'\n✅ {len(source_texts)} source(s) loaded')

In [ ]:
# SUMMARIZE
print('🧠 Summarizing...')
summaries = []
for i, src in enumerate(source_texts):
    summary = query_llm(f'Summarize in 2-3 short paragraphs for a video essay about "{TOPIC}":\n\n{src["text"][:6000]}', system_prompt='You are a research assistant. Be concise.')
    summaries.append({'source': src['source'], 'summary': summary})
    print(f'   Source {i+1}: ✅ ({len(summary)} chars)')
with open(WORK_DIR / 'summaries.json', 'w') as f: json.dump({'topic': TOPIC, 'sources': summaries}, f, indent=2)
print('✅ Saved to Drive')

In [ ]:
# VIEW
for i, s in enumerate(summaries):
    display(Markdown(f'### Source {i+1}: {s["source"]}'))
    display(Markdown(s['summary']))
    display(Markdown('---'))

---
## ✍️ Station 2: The Script Writer

> **🤔 Think about:** Who is the "author" here?

In [ ]:
# GENERATE ESSAY OUTLINE
print('🧠 Generating outline...')
summaries_text = '\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])

outline_prompt = f'''Create an essay outline for a SHORT video essay (~{TARGET_VIDEO_LENGTH} seconds) about: "{TOPIC}"

RESEARCH:
{summaries_text}

REQUIREMENTS:
- Exactly {NUM_CHAPTERS} chapters
- This is a SHORT video ({TARGET_VIDEO_LENGTH}s total). Keep it focused and tight.
- Each chapter should make ONE clear point

Format as JSON:
{{"title": "...", "thesis": "...", "chapters": [{{"chapter": 1, "title": "...", "summary": "...", "key_points": ["..."]}}]}}

Only output JSON.'''

for attempt in range(3):
    try:
        outline = parse_json_response(query_llm(outline_prompt, 'You are an expert essay writer. Create compelling, concise video essay scripts.'))
        break
    except Exception as e:
        print(f'   Attempt {attempt+1}/3 failed: {e}')
        if attempt == 2: raise
if len(outline.get('chapters',[])) > NUM_CHAPTERS: outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]
print(f'\n📖 {outline.get("title", "Untitled")}')
for ch in outline.get('chapters',[]): print(f'   Ch {ch["chapter"]}: {ch["title"]}')

In [ ]:
# WRITE CHAPTERS — with strict time budget per scene
print('✍️ Writing chapters...')

chapter_system = f'''You are a video essay scriptwriter writing for a SHORT {TARGET_VIDEO_LENGTH}-second video.

CRITICAL TIME CONSTRAINTS:
- Each section will become a {SECONDS_PER_SCENE}-second video scene
- Narration must be VERY SHORT: maximum {int(WORDS_PER_SCENE)} words per section
- That is 1-2 SHORT sentences. NOT more.
- Think of it like a TikTok voiceover — every word counts
- Visual directions should describe what happens in {SECONDS_PER_SCENE} seconds

EXAMPLE of correct length narration (for a 5-second scene):
"The first AI artwork sold at auction shocked the art world. A portrait generated by algorithms fetched $432,000."
That is 20 words = ~5 seconds of speech. This is the TARGET length.'''

for i, chapter in enumerate(outline['chapters']):
    print(f'\n📝 Chapter {chapter["chapter"]}: {chapter["title"]}')
    chapter_prompt = f'''Write sections for this chapter of a {TARGET_VIDEO_LENGTH}-second video:

TOPIC: {TOPIC}
THESIS: {outline.get('thesis', '')}
CHAPTER {chapter['chapter']}: {chapter['title']}
Summary: {chapter.get('summary', '')}

CRITICAL: Each section's narration must be MAX {int(WORDS_PER_SCENE)} words (1-2 short sentences).
Write exactly {SECTIONS_PER_CHAPTER} sections as JSON array:
[{{"section": 1, "narration": "MAX {int(WORDS_PER_SCENE)} WORDS HERE", "visual_direction": "what to show in {SECONDS_PER_SCENE}s", "mood": "..."}}]

Only output JSON.'''

    try:
        sections = parse_json_response(query_llm(chapter_prompt, chapter_system))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except:
        sections = [{'section': 1, 'narration': chapter['title'], 'visual_direction': chapter.get('summary',''), 'mood': 'neutral'}]
    if len(sections) > SECTIONS_PER_CHAPTER: sections = sections[:SECTIONS_PER_CHAPTER]
    
    # Enforce word limit
    for s in sections:
        words = s.get('narration', '').split()
        if len(words) > int(WORDS_PER_SCENE * 1.5):  # Allow 50% overflow max
            s['narration'] = ' '.join(words[:int(WORDS_PER_SCENE)]) + '.'
            print(f'   ⚠️ Trimmed narration to {int(WORDS_PER_SCENE)} words')
    
    outline['chapters'][i]['sections'] = sections
    for s in sections:
        wc = len(s.get('narration','').split())
        print(f'   Sec {s.get("section","?")}: {s.get("narration","")[:60]}... ({wc} words)')

outline['credits'] = {'sources': [s['source'] for s in summaries]}
with open(WORK_DIR / 'essay_script.json', 'w') as f: json.dump(outline, f, indent=2)

total_words = sum(len(s.get('narration','').split()) for ch in outline['chapters'] for s in ch.get('sections',[]))
est_duration = total_words / 2.5
print(f'\n✅ Essay saved! {total_words} words total ≈ {est_duration:.0f}s narration (target: {TARGET_VIDEO_LENGTH}s video)')

In [ ]:
# VIEW ESSAY
display(Markdown(f'# {outline.get("title", "Untitled")}'))
display(Markdown(f'*{outline.get("thesis", "")}*\n\n---'))
for ch in outline['chapters']:
    display(Markdown(f'## Chapter {ch["chapter"]}: {ch["title"]}'))
    for s in ch.get('sections', []):
        wc = len(s.get('narration','').split())
        display(Markdown(f'**Sec {s.get("section","?")}** — *{s.get("mood","?")}* ({wc} words ≈ {wc/2.5:.0f}s)\n\n> 🎙️ {s.get("narration","")}\n\n> 🎬 {s.get("visual_direction","")}\n\n---'))

---
## 🎬 Station 3: Visual Translation

> **🤔 Think about:** What aesthetic assumptions are baked into style categories like "documentary" or "horror"?

In [ ]:
# STYLE ARC
STYLE_ARCS = {
    'calm_to_dramatic': {'description': 'Starts calm, builds to intense', 'early': ['documentary','nature'], 'middle': ['news_report','sports_commentary'], 'late': ['action_movie','horror'], 'credits': ['documentary']},
    'documentary_journey': {'description': 'Classic documentary, increasing energy', 'early': ['documentary'], 'middle': ['nature','news_report','documentary'], 'late': ['action_movie','sports_commentary'], 'credits': ['documentary']},
    'chaos_arc': {'description': 'Chaotic brainrot energy', 'early': ['classic_brainrot','zoomer_brainrot'], 'middle': ['surreal_brainrot','hyperpop_brainrot'], 'late': ['fever_dream_brainrot','cursed_brainrot'], 'credits': ['documentary']},
}
CONTENT_STYLES = {
    'documentary': {'visual_keywords': 'cinematic, professional lighting', 'audio_keywords': 'ambient sounds'},
    'nature': {'visual_keywords': 'wide nature shots, golden hour', 'audio_keywords': 'nature sounds, wind'},
    'news_report': {'visual_keywords': 'news studio, professional framing', 'audio_keywords': 'news audio, serious'},
    'sports_commentary': {'visual_keywords': 'dynamic angles, slow motion', 'audio_keywords': 'crowd, energetic'},
    'action_movie': {'visual_keywords': 'dynamic movement, dramatic lighting', 'audio_keywords': 'orchestral hits'},
    'horror': {'visual_keywords': 'dark lighting, shadows, fog', 'audio_keywords': 'creepy ambience, drones'},
    'classic_brainrot': {'visual_keywords': 'chaotic, deep fried', 'audio_keywords': 'bass boosted, vine booms'},
    'surreal_brainrot': {'visual_keywords': 'dreamlike, impossible geometry', 'audio_keywords': 'slowed reverb'},
    'zoomer_brainrot': {'visual_keywords': 'meme aesthetic, ironic', 'audio_keywords': 'TikTok sounds'},
    'hyperpop_brainrot': {'visual_keywords': 'maximalist, Y2K, chrome', 'audio_keywords': 'hyperpop beats'},
    'fever_dream_brainrot': {'visual_keywords': 'hallucinatory, warped', 'audio_keywords': 'echoing, stretched'},
    'cursed_brainrot': {'visual_keywords': 'unsettling, jpeg artifacts', 'audio_keywords': 'distorted'},
    'none': {'visual_keywords': '', 'audio_keywords': ''}
}

CHOSEN_ARC = 'calm_to_dramatic'  # Options: calm_to_dramatic, documentary_journey, chaos_arc
arc = STYLE_ARCS[CHOSEN_ARC]
print(f'🎨 Style Arc: {CHOSEN_ARC} — {arc["description"]}')

In [ ]:
# STORYBOARD + STYLES
print('🎬 Generating storyboard...')
scenes = []; sn = 1
scenes.append({'scene': sn, 'scene_type': 'title_card', 'title': outline.get('title','Untitled'), 'description': outline.get('thesis','')}); sn += 1
for ch in outline.get('chapters', []):
    for sec in ch.get('sections', []):
        scenes.append({'scene': sn, 'scene_type': 'content', 'title': f"{ch['title']} - Part {sec.get('section',1)}", 'narration': sec.get('narration',''), 'visual_direction': sec.get('visual_direction',''), 'mood': sec.get('mood',''), 'chapter': ch.get('chapter',0)}); sn += 1
scenes.append({'scene': sn, 'scene_type': 'credits', 'title': 'Credits', 'description': ', '.join(outline.get('credits',{}).get('sources',[]))})

content_scenes = [s for s in scenes if s['scene_type']=='content']
total=len(content_scenes); early_end=max(1,int(total*0.25)); late_start=max(early_end+1,int(total*0.75))
last_style=None
for i,sc in enumerate(content_scenes):
    pool = arc.get('early' if i<early_end else 'late' if i>=late_start else 'middle', ['documentary'])
    available=[s for s in pool if s!=last_style] or pool; style=random.choice(available)
    sc['assigned_style']=style; sc['visual_keywords']=CONTENT_STYLES.get(style,{}).get('visual_keywords',''); sc['audio_keywords']=CONTENT_STYLES.get(style,{}).get('audio_keywords',''); last_style=style

with open(WORK_DIR/'storyboard.json','w') as f: json.dump(scenes,f,indent=2)
for s in scenes:
    tag=f' [{s.get("assigned_style","")}]' if s.get('assigned_style') else ''
    print(f'   Scene {s["scene"]}: [{s["scene_type"]}] {s["title"]}{tag}')

In [ ]:
# T2V PROMPTS
print('🎥 Converting to video prompts...')
OPENINGS = ['Start with SHOT TYPE','Start with ACTION','Start with ENVIRONMENT','Start with LIGHTING','Start with CAMERA MOVEMENT']
prompts = []
for sc in scenes:
    if sc['scene_type']!='content': continue
    style=sc.get('assigned_style','none'); vk=sc.get('visual_keywords',''); ak=sc.get('audio_keywords','')
    style_inst=f'\nStyle: {style}\nVisual: {vk}\nAudio: {ak}' if style!='none' else ''
    sys_p=f'You are an expert at text-to-video prompts. Write ONE paragraph, 3-5 sentences for a {SECONDS_PER_SCENE}-second clip: shot type, setting, action, camera, audio.{style_inst}\nNo text/writing. Max 2 subjects.'
    user_p=f'Convert to T2V prompt for a {SECONDS_PER_SCENE}-second clip:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n\n{random.choice(OPENINGS)}\n\nOutput ONLY the prompt.'
    print(f'   Scene {sc["scene"]}: [{style}]...', end=' ')
    t2v=query_llm(user_p,sys_p).strip().strip('"')
    prompts.append({'scene':sc['scene'],'title':sc['title'],'t2v_prompt':t2v,'style':style,'narration':sc.get('narration',''),'duration':SECONDS_PER_SCENE})
    print('✅')

with open(WORK_DIR/'prompts.json','w') as f: json.dump(prompts,f,indent=2)
est_total = len(prompts) * SECONDS_PER_SCENE
print(f'\n✅ {len(prompts)} prompts saved ({est_total}s estimated video)')

In [ ]:
# VIEW PROMPTS
display(Markdown(f'# 🎬 Video Production Plan\n*{len(prompts)} scenes × ~{SECONDS_PER_SCENE}s = ~{len(prompts)*SECONDS_PER_SCENE}s*\n\n---'))
for p in prompts:
    wc = len(p['narration'].split())
    display(Markdown(f'### Scene {p["scene"]}: {p["title"]}\n**Style:** `{p["style"]}` | **Duration:** {p["duration"]}s | **Narration:** {wc} words (~{wc/2.5:.0f}s)\n\n> 🎙️ {p["narration"]}\n\n> 🎥 {p["t2v_prompt"]}\n\n---'))
print('📁 Files saved to Google Drive/rotbots_workshop/')
print('   → Open 04_The_Voice.ipynb or 05_Generate.ipynb next!')

---
## ⏭️ Next Steps

All files on Google Drive. Open **04_The_Voice.ipynb** for narration or **05_Generate.ipynb** for video clips.

---
*ROTBOTS Workshop — LI-MA Transformation Digital Art 2026*